<a href="https://colab.research.google.com/github/Sayak-coder/HateSpeech-detection/blob/main/Subtask%20(B)%20%5BIT%5D%20Hate%20Speech%20Classification%20part2..ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Experiment 2.A
only comment as input

In [35]:
!pip install transformers

In [17]:
import pandas as pd
import numpy as np
import torch

In [18]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [19]:
df_IT = pd.read_csv("/content/StereoQueerEval_IT_training.tsv", sep='\t')
df_IT.head()

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target
0,training_IT_0001,"Nivan transgender a 17 anni: ""Il mio percorso ...",Nivan è un ragazzo transgender Ftm di soli 17 ...,Sono transgender ma ho paura di dirlo ai miei ...,no,no,none
1,training_IT_0002,"Casillas annuncia di essere gay, poi parla di ...","Travolto dalle polemiche, Casillas è stato cos...",E cosa sarà mai guardate cosa succede nel mond...,no,yes_implicit,group_g
2,training_IT_0003,Schlein: «Con governo Meloni l’Italia è scivol...,«Sono sempre felice quando c’è il riconoscimen...,Ma fate silenzio il gay pride e finito ora and...,yes,yes_explicit,group_lgbtqia+
3,training_IT_0004,"Olimpiadi Parigi 2024, la pugile Imane Khelif ...",Dopo che Angela Carini alle Olimpiadi di Parig...,Donna Trans NON ESISTE! Un uomo non puo` diven...,yes,yes_explicit,group_t
4,training_IT_0005,"Roma Pride 2018, l'inno ""Ma che ti importa"": ""...",Sabato 9 giugno il Roma Pride scende in piazza...,Allora che vi importa se uno dice che prenderl...,no,yes_explicit,"group_g,lgbtqia+"


In [20]:
df = df_IT

In [27]:
df.shape

(2400, 7)

In [26]:
df['hate_speech'].value_counts()

,count
hate_speech,
no,1160
yes_explicit,659
yes_implicit,581


In [30]:
df.isnull().sum()

,0
StereoQueerEval_id,0
yt_title,0
yt_description,0
yt_comment,0
stereotype,0
hate_speech,0
target,0


In [21]:
df.columns

Index(['StereoQueerEval_id', 'yt_title', 'yt_description', 'yt_comment',
       'stereotype', 'hate_speech', 'target'],
      dtype='object')

In [33]:
X = list(df['yt_comment'])
y = df['hate_speech'].map({'no':0, 'yes_explicit':1, 'yes_implicit':2}).tolist()

In [34]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 45)
X_val, X_test, y_val, y_test = train_test_split(X, y, test_size = 0.5, random_state = 45)

In [40]:
def build_inputs(d):
    comments = list(d['yt_comment'])
    return comments

def encode(d, tokenizer, max_length=256):
    comments = build_inputs(d)
    return tokenizer(comments, truncation=True, max_length=max_length)

In [37]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

### twitter-roberta-base-hate-latest

In [38]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [43]:
train_encodings = tokenizer(X_train, truncation=True, max_length=256)
val_encodings = tokenizer(X_val, truncation=True, max_length=256)
test_encodings  = tokenizer(X_test,  truncation=True, max_length=256)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset  = TextDataset(test_encodings,  y_test)

In [47]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [48]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "explicit", 2: "implicit"},
    label2id={"no": 0, "explicit": 1, "implicit": 2},
).to(device)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-hate-latest
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [49]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.102257
20,1.079588
30,1.061334
40,1.071141
50,0.986640
60,1.012373
70,1.040949
80,1.022423
90,1.069299
100,1.012915


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.8200952506065369, metrics={'train_runtime': 360.019, 'train_samples_per_second': 26.665, 'train_steps_per_second': 1.667, 'total_flos': 921505411826496.0, 'train_loss': 0.8200952506065369, 'epoch': 5.0})

In [50]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","explicit","implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.763     0.797     0.780       587
    explicit      0.694     0.663     0.679       315
    implicit      0.577     0.554     0.565       298

    accuracy                          0.702      1200
   macro avg      0.678     0.671     0.675      1200
weighted avg      0.699     0.702     0.700      1200

[[468  48  71]
 [ 56 209  50]
 [ 89  44 165]]


### mdeberta-v3-base

In [51]:
from transformers import AutoTokenizer

MODEL = f"microsoft/mdeberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL)


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spm.model: downloading bytes:           |  0.00B            

In [52]:
train_encodings = tokenizer(X_train, truncation=True, max_length=256)
val_encodings = tokenizer(X_val, truncation=True, max_length=256)
test_encodings  = tokenizer(X_test,  truncation=True, max_length=256)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset  = TextDataset(test_encodings,  y_test)

In [53]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=75,
    learning_rate=1e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=10
)


In [54]:
from transformers import AutoModelForSequenceClassification

MODEL = f"microsoft/mdeberta-v3-base"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "explicit", 2: "implicit"},
    label2id={"no": 0, "explicit": 1, "implicit": 2},
).to(device)


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.33GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight   

In [55]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_mdeberta_v3_base = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_mdeberta_v3_base.train()

Step,Training Loss
10,1.083851
20,1.075511
30,1.072584
40,1.066965
50,1.019269
60,1.032655
70,1.062376
80,1.115325
90,1.113993
100,1.065714


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.5164599720637003, metrics={'train_runtime': 228.7636, 'train_samples_per_second': 41.965, 'train_steps_per_second': 2.623, 'total_flos': 824858912479680.0, 'train_loss': 0.5164599720637003, 'epoch': 5.0})

In [56]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_mdeberta_v3_base.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","explicit","implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.489     1.000     0.657       587
    explicit      0.000     0.000     0.000       315
    implicit      0.000     0.000     0.000       298

    accuracy                          0.489      1200
   macro avg      0.163     0.333     0.219      1200
weighted avg      0.239     0.489     0.321      1200

[[587   0   0]
 [315   0   0]
 [298   0   0]]


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


###distilbert

In [57]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [58]:
train_encodings = tokenizer(X_train, truncation=True, max_length=256)
val_encodings = tokenizer(X_val, truncation=True, max_length=256)
test_encodings  = tokenizer(X_test,  truncation=True, max_length=256)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset  = TextDataset(test_encodings,  y_test)

In [59]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [60]:
from transformers import DistilBertForSequenceClassification

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes_explicit", 2:"yes_implicit"},
    label2id={"no": 0, "yes_explicit": 1, "yes_implicit": 2}
).to(device)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [61]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,1.083206
20,1.075468
30,1.038152
40,1.130874
50,1.099462
60,1.029460
70,1.071151
80,1.088782
90,1.006986
100,0.998985


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1200, training_loss=0.7980057080586751, metrics={'train_runtime': 214.5255, 'train_samples_per_second': 44.75, 'train_steps_per_second': 5.594, 'total_flos': 350844092232048.0, 'train_loss': 0.7980057080586751, 'epoch': 5.0})

In [62]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes_explicit","yes_implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.796     0.833     0.814       587
yes_explicit      0.668     0.733     0.699       315
yes_implicit      0.621     0.500     0.554       298

    accuracy                          0.724      1200
   macro avg      0.695     0.689     0.689      1200
weighted avg      0.719     0.724     0.719      1200

[[489  44  54]
 [ 47 231  37]
 [ 78  71 149]]


###Experiment 2.B
comment + description as input

In [63]:
X = (df.drop(['StereoQueerEval_id', 'yt_description', 'stereotype', 'target', 'hate_speech'], axis=1))
y = df['hate_speech'].map({'no':0, 'yes_explicit':1, 'yes_implicit':2})

In [64]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=45)
X_val,X_test,y_val,y_test = train_test_split(X_test,y_test,test_size=0.5,random_state=45)

In [65]:
def build_inputs(d):
    comments = d['yt_comment'].tolist()
    title = d['yt_title'].fillna("")
    ctx = title
    return comments, ctx.tolist()

def encode(d, tokenizer, max_length=256):
    comments, ctx = build_inputs(d)
    return tokenizer(comments, ctx, truncation="longest_first", max_length=max_length)

In [66]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

###twitter-roberta-base-hate-latest

In [68]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [69]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [73]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [74]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "explicit", 2: "implicit"},
    label2id={"no": 0, "explicit": 1, "implicit": 2},
).to(device)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-hate-latest
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [75]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.105350
20,1.083945
30,1.044888
40,1.072824
50,0.933662
60,0.974568
70,1.022368
80,1.011123
90,0.988752
100,0.922666


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.8390817848841349, metrics={'train_runtime': 437.5766, 'train_samples_per_second': 21.939, 'train_steps_per_second': 1.371, 'total_flos': 1037612429693280.0, 'train_loss': 0.8390817848841349, 'epoch': 5.0})

In [76]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","explicit","implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.768     0.803     0.785       132
    explicit      0.431     0.574     0.492        54
    implicit      0.333     0.185     0.238        54

    accuracy                          0.613       240
   macro avg      0.511     0.521     0.505       240
weighted avg      0.594     0.613     0.596       240

[[106  14  12]
 [ 15  31   8]
 [ 17  27  10]]


###distilbert-base-multilingual-cased

In [77]:
from transformers import AutoTokenizer

MODEL = f"distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [78]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [79]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [81]:
from transformers import DistilBertForSequenceClassification

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes_explicit", 2:"yes_implicit"},
    label2id={"no": 0, "yes_explicit": 1, "yes_implicit": 2}
).to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [82]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,1.124960
20,1.100479
30,1.112278
40,1.101204
50,1.078411
60,1.028767
70,1.061470
80,1.063570
90,0.952269
100,0.924012


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1200, training_loss=0.776470551888148, metrics={'train_runtime': 265.5617, 'train_samples_per_second': 36.15, 'train_steps_per_second': 4.519, 'total_flos': 347791078208448.0, 'train_loss': 0.776470551888148, 'epoch': 5.0})

In [83]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes_explicit","yes_implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.744     0.682     0.711       132
yes_explicit      0.382     0.481     0.426        54
yes_implicit      0.412     0.389     0.400        54

    accuracy                          0.571       240
   macro avg      0.513     0.517     0.513       240
weighted avg      0.588     0.571     0.577       240

[[90 25 17]
 [15 26 13]
 [16 17 21]]


###distlbert-English

In [84]:
from transformers import DistilBertTokenizerFast
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased-finetuned-sst-2-english')

In [85]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [90]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [91]:
from transformers import DistilBertForSequenceClassification

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes_explicit", 2:"yes_implicit"},
    label2id={"no": 0, "yes_explicit": 1, "yes_implicit": 2}
).to(device)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [92]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,1.080097
20,1.061153
30,1.047631
40,1.105046
50,1.082140
60,1.057299
70,1.067711
80,1.061438
90,0.972484
100,0.969476


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1200, training_loss=0.8239044392108917, metrics={'train_runtime': 210.9921, 'train_samples_per_second': 45.499, 'train_steps_per_second': 5.687, 'total_flos': 417577804187904.0, 'train_loss': 0.8239044392108917, 'epoch': 5.0})

In [93]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes_explicit","yes_implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.748     0.697     0.722       132
yes_explicit      0.364     0.519     0.427        54
yes_implicit      0.325     0.241     0.277        54

    accuracy                          0.554       240
   macro avg      0.479     0.485     0.475       240
weighted avg      0.566     0.554     0.555       240

[[92 23 17]
 [16 28 10]
 [15 26 13]]


###Experiment 2.C
comment + description + title as input

In [94]:
X = (df.drop(['StereoQueerEval_id', 'stereotype', 'target', 'hate_speech'], axis=1))
y = df['hate_speech'].map({"no": 0, "yes_explicit": 1, "yes_implicit": 2})

In [95]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=45)
X_val,X_test,y_val,y_test = train_test_split(X_test,y_test,test_size=0.5,random_state=45)

In [96]:
def build_inputs(d):
    comments = d['yt_comment'].tolist()
    title = d['yt_title'].fillna("")
    desc  = d['yt_description'].fillna("").str[:300]        # trim long descriptions
    ctx = (title + " . " + desc)
    return comments, ctx.tolist()

def encode(d, tokenizer, max_length=256):
    comments, ctx = build_inputs(d)
    return tokenizer(comments, ctx, truncation="longest_first", max_length=max_length)

In [97]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

###cardiffnlp/twitter-roberta-base-hate-latest

In [98]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [99]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [100]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)


In [101]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=2,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes_explicit", 2:"yes_implicit"},
    label2id={"no": 0, "yes_explicit": 1, "yes_implicit": 2},
).to(device)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-hate-latest
Key                        | Status   |                                                                                       
---------------------------+----------+---------------------------------------------------------------------------------------
classifier.out_proj.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2]) vs model:torch.Size([3])          
classifier.out_proj.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([2, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [102]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.075990
20,1.075846
30,1.047116
40,1.025242
50,0.949320
60,0.953012
70,0.999666
80,0.956921
90,0.983178
100,0.929364


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=600, training_loss=0.8630184030532837, metrics={'train_runtime': 514.0319, 'train_samples_per_second': 18.676, 'train_steps_per_second': 1.167, 'total_flos': 1248530723179488.0, 'train_loss': 0.8630184030532837, 'epoch': 5.0})

In [103]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes_explicit","yes_implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.795     0.735     0.764       132
yes_explicit      0.403     0.537     0.460        54
yes_implicit      0.391     0.333     0.360        54

    accuracy                          0.600       240
   macro avg      0.530     0.535     0.528       240
weighted avg      0.616     0.600     0.605       240

[[97 19 16]
 [13 29 12]
 [12 24 18]]


###distilbert-Multilingual

In [104]:
from transformers import AutoTokenizer

MODEL = f"distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [105]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [106]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [107]:
from transformers import DistilBertForSequenceClassification

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes_explicit", 2:"yes_implicit"},
    label2id={"no": 0, "yes_explicit": 1, "yes_implicit": 2}
).to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [108]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,1.102151
20,1.106674
30,1.088255
40,1.104162
50,1.083800
60,1.043897
70,1.072293
80,1.077686
90,0.963441
100,0.949989


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1200, training_loss=0.8254885947704316, metrics={'train_runtime': 362.3327, 'train_samples_per_second': 26.495, 'train_steps_per_second': 3.312, 'total_flos': 508990218654528.0, 'train_loss': 0.8254885947704316, 'epoch': 5.0})

In [109]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes_explicit","yes_implicit"], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.767     0.697     0.730       132
yes_explicit      0.378     0.519     0.438        54
yes_implicit      0.435     0.370     0.400        54

    accuracy                          0.583       240
   macro avg      0.527     0.529     0.523       240
weighted avg      0.605     0.583     0.590       240

[[92 25 15]
 [15 28 11]
 [13 21 20]]
